In [ ]:
# =============================================================================
# Etape 4 : deux images couplees (boucle T1/T2) | CELLULE UNIQUE : Execution -> Tout executer (GPU T4)
# Duree estimee : ~1 h 30 a 2 h sur T4 (S2 et S3, 3 graines, 2 modeles a boucle ; T_rope et V_lam0 sont deja dans le zip du depot et sautes).
# Au premier lancement, un BOUTON de televersement apparait : choisir le zip du depot
# (GitHub -> Code -> Download ZIP, branche autoencodeur-visuel). En cas de reprise,
# choisir AUSSI results_st4.zip (selection multiple) : les mesures deja faites sont sautees.
# Toutes les heures et a la fin, results_st4.zip est TELECHARGE dans ton navigateur :
# c'est la sauvegarde ; la joindre a la discussion pour lire les resultats.
# =============================================================================
import os
import shutil
import sys
import time
import zipfile

import torch

print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN (Execution -> Modifier le type d'execution -> GPU T4)")


def deploy():
    """Deploie le depot depuis le zip televerse ; fusionne results_st4.zip. Retourne la racine du depot."""
    cwd0 = os.getcwd()
    if os.path.exists(os.path.join("src", "ae_run.py")):
        return cwd0
    zips = sorted(f for f in os.listdir(cwd0) if f.endswith(".zip"))
    if not zips:
        try:
            from google.colab import files as colab_files
            print("Clique sur le bouton ci-dessous et televerse le zip du depot (plus results_st4.zip pour reprendre).")
            colab_files.upload()
            zips = sorted(f for f in os.listdir(cwd0) if f.endswith(".zip"))
        except ImportError:
            pass
    assert zips, "Aucun zip present : televerse le zip du depot (GitHub -> Code -> Download ZIP)."
    repo, res = None, []
    os.makedirs("_unzip", exist_ok=True)
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            names = zf.namelist()
            if any(n.endswith("src/s3_tasks.py") for n in names):
                dest = os.path.join("_unzip", z[:-4])
                zf.extractall(dest)
                for root, dirs, fs in os.walk(dest):
                    if os.path.exists(os.path.join(root, "src", "ae_run.py")):
                        repo = os.path.abspath(root)
            elif any(n.startswith("results/st_") for n in names):
                res.append(z)
    assert repo, "Le zip du depot (contenant src/s3_tasks.py) est introuvable : verifie que c'est la branche autoencodeur-visuel."
    for z in res:
        with zipfile.ZipFile(z) as zf:
            zf.extractall(repo)
        print("Resultats precedents fusionnes :", z)
    return repo


ROOT = deploy()
os.chdir(ROOT)
sys.path.insert(0, os.path.join(ROOT, "src"))
print("Depot :", ROOT)

_last = [0.0]


def save_and_download(force=False):
    """Reconstruit results_st4.zip et le telecharge (toutes les heures, ou a la fin)."""
    if not force and time.time() - _last[0] < 3600:
        return
    _last[0] = time.time()
    base = os.path.join("/content" if os.path.isdir("/content") else ROOT, "results_st4")
    shutil.make_archive(base, "zip", ROOT, "results")
    try:
        from google.colab import files as colab_files
        colab_files.download(base + ".zip")
    except ImportError:
        print("Sauvegarde :", base + ".zip")


import s3_tasks

_last[0] = time.time()
s3_tasks.run_all(ROOT, st=s3_tasks.ST_LOOP, tasks=s3_tasks.TASKS_LOOP, on_done=save_and_download)
save_and_download(force=True)

# Resume
print(open(os.path.join(ROOT, 'results', 'st4_summary.md'), encoding='utf-8').read())
